## Customer Churn Prediction with an Artificial Neural Network

This project predicts which telecom customers will leave the company (churn) with an artificial neural network (ANN). Because only about 27% of the customers leave, we pay special attention to finding the customers who will leave, and we compare the network with classic models. The objective is in order to find the customers who are likely to leave so that the company can offer them a retention campaign, and understand what characterizes them.


## Approach
1. Load the data, explore it, and look at how the churn rate changes by contract type, tenure, and charges
2. Clean the data (fix the TotalCharges column, remove the customer ID)
3. Convert the categorical columns to numbers (one-hot encoding), split the data, and scale the numeric columns using only the training data
4. Build and train an ANN with two hidden layers, dropout, class weights, and early stopping
5. Evaluate it with recall, precision, F1, AUC, and a confusion matrix, and compare it with Logistic Regression and Random Forest
6. Choose a decision threshold that fits the business goal
7. Save the model for a simple churn risk calculator
8. Publish in Streamlit

In [1]:
# Load the data
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

In [2]:
df.shape

(7043, 21)

In [3]:
df.head(3)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [5]:
df["Churn"].value_counts(normalize=True).round(3)

,proportion
Churn,
No,0.735
Yes,0.265


In [6]:
df.groupby("Contract")["Churn"].apply(lambda s: (s == "Yes").mean()).round(3)

,Churn
Contract,
Month-to-month,0.427
One year,0.113
Two year,0.028


In [7]:
# We don't need the customerID
df = df.drop(columns="customerID")

In [8]:
# Convert the numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [9]:
df["TotalCharges"].isnull().sum()

np.int64(11)

In [10]:
df.loc[df["TotalCharges"].isnull(), ["tenure", "MonthlyCharges", "TotalCharges"]].head(3)

,tenure,MonthlyCharges,TotalCharges
488,0,52.55,NaN
753,0,20.25,NaN
936,0,80.85,NaN


In [11]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [13]:
y = (df.pop("Churn") == "Yes").astype(int).values
x = pd.get_dummies(df, drop_first=True).astype("float32")

In [14]:
x.shape

(7043, 30)

In [15]:
# Split the data
from sklearn.model_selection import train_test_split

xtr, xte, ytr, yte = train_test_split(x, y, test_size=0.20, random_state=42, stratify=y)

In [16]:
xtr.shape, xte.shape, ytr.mean().round(3), yte.mean().round(3)

((5634, 30), (1409, 30), np.float64(0.265), np.float64(0.265))

In [17]:
# Scaling
from sklearn.preprocessing import StandardScaler

num = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler().fit(xtr[num])
xtr, xte = xtr.copy(), xte.copy()
xtr[num] = scaler.transform(xtr[num])
xte[num] = scaler.transform(xte[num])

In [18]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = dict(enumerate(compute_class_weight("balanced", classes=np.array([0, 1]), y=ytr)))
print(class_weights)

{0: np.float64(0.6805991785455424), 1: np.float64(1.8842809364548494)}


In [19]:
# Train ANN
def train_ann(use_weights):
    tf.keras.utils.set_random_seed(42)
    model = tf.keras.Sequential([
        layers.Input((30,)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(16, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
    model.fit(xtr.values, ytr, epochs=60, batch_size=64, validation_split=0.15,
              class_weight=class_weights if use_weights else None, callbacks=[stop], verbose=2)
    return model

In [20]:
ann_plain = train_ann(use_weights=False)

Epoch 1/60
75/75 - 4s - 48ms/step - accuracy: 0.6997 - auc: 0.6190 - loss: 0.5920 - val_accuracy: 0.7730 - val_auc: 0.7778 - val_loss: 0.4858
Epoch 2/60
75/75 - 1s - 8ms/step - accuracy: 0.7711 - auc: 0.7870 - loss: 0.4751 - val_accuracy: 0.7742 - val_auc: 0.8021 - val_loss: 0.4590
Epoch 3/60
75/75 - 0s - 6ms/step - accuracy: 0.7845 - auc: 0.8085 - loss: 0.4567 - val_accuracy: 0.7719 - val_auc: 0.8069 - val_loss: 0.4545
Epoch 4/60
75/75 - 0s - 6ms/step - accuracy: 0.7849 - auc: 0.8146 - loss: 0.4514 - val_accuracy: 0.7742 - val_auc: 0.8086 - val_loss: 0.4534
Epoch 5/60
75/75 - 0s - 4ms/step - accuracy: 0.7905 - auc: 0.8222 - loss: 0.4451 - val_accuracy: 0.7825 - val_auc: 0.8109 - val_loss: 0.4497
Epoch 6/60
75/75 - 0s - 5ms/step - accuracy: 0.7897 - auc: 0.8240 - loss: 0.4429 - val_accuracy: 0.7790 - val_auc: 0.8133 - val_loss: 0.4482
Epoch 7/60
75/75 - 0s - 6ms/step - accuracy: 0.7920 - auc: 0.8300 - loss: 0.4348 - val_accuracy: 0.7813 - val_auc: 0.8134 - val_loss: 0.4483
Epoch 8/60
7

In [21]:
ann_balanced = train_ann(use_weights=True)

Epoch 1/60
75/75 - 4s - 47ms/step - accuracy: 0.5909 - auc: 0.6922 - loss: 0.6375 - val_accuracy: 0.6832 - val_auc: 0.7936 - val_loss: 0.6147
Epoch 2/60
75/75 - 1s - 9ms/step - accuracy: 0.7189 - auc: 0.7976 - loss: 0.5500 - val_accuracy: 0.6950 - val_auc: 0.8042 - val_loss: 0.5655
Epoch 3/60
75/75 - 1s - 9ms/step - accuracy: 0.7356 - auc: 0.8128 - loss: 0.5308 - val_accuracy: 0.6939 - val_auc: 0.8099 - val_loss: 0.5610
Epoch 4/60
75/75 - 0s - 5ms/step - accuracy: 0.7402 - auc: 0.8185 - loss: 0.5242 - val_accuracy: 0.6986 - val_auc: 0.8121 - val_loss: 0.5451
Epoch 5/60
75/75 - 0s - 4ms/step - accuracy: 0.7414 - auc: 0.8242 - loss: 0.5187 - val_accuracy: 0.6986 - val_auc: 0.8137 - val_loss: 0.5393
Epoch 6/60
75/75 - 0s - 4ms/step - accuracy: 0.7299 - auc: 0.8257 - loss: 0.5139 - val_accuracy: 0.7021 - val_auc: 0.8148 - val_loss: 0.5378
Epoch 7/60
75/75 - 0s - 4ms/step - accuracy: 0.7435 - auc: 0.8317 - loss: 0.5056 - val_accuracy: 0.7021 - val_auc: 0.8150 - val_loss: 0.5379
Epoch 8/60
7

In [22]:
# Evaluation - Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

lr = LogisticRegression(max_iter=2000).fit(xtr, ytr)
lr_bal = LogisticRegression(max_iter=2000, class_weight="balanced").fit(xtr, ytr)

In [23]:
probs = {
    "ANN": ann_plain.predict(xte.values, verbose=0).ravel(),
    "ANN (balanced)": ann_balanced.predict(xte.values, verbose=0).ravel(),
    "LogReg": lr.predict_proba(xte)[:, 1],
    "LogReg (balanced)": lr_bal.predict_proba(xte)[:, 1],
}

In [24]:
#Comparition
def scores(p):
    pred = p > 0.5
    return pd.Series({"accuracy": accuracy_score(yte, pred), "churn recall": recall_score(yte, pred), "AUC": roc_auc_score(yte, p)})

pd.DataFrame({n: scores(p) for n, p in probs.items()}).T.round(3)

,accuracy,churn recall,AUC
ANN,0.793,0.535,0.839
ANN (balanced),0.737,0.797,0.839
LogReg,0.806,0.559,0.842
LogReg (balanced),0.738,0.783,0.842


In [28]:
from sklearn.metrics import precision_score
p = probs["ANN"]
pd.DataFrame([{"threshold": round(t, 2), "flagged customers": int((p > t).sum()),
               "churn recall": recall_score(yte, p > t), "churn precision": precision_score(yte, p > t)}
              for t in np.arange(0.2, 0.65, 0.05)]).round(3)

,threshold,flagged customers,churn recall,churn precision
0,0.20,709,0.866,0.457
1,0.25,644,0.834,0.484
2,0.30,567,0.778,0.513
3,0.35,508,0.727,0.535
4,0.40,455,0.695,0.571
5,0.45,388,0.623,0.601
6,0.50,317,0.535,0.631
7,0.55,261,0.471,0.674
8,0.60,200,0.377,0.705


In [29]:
#  Preparation for Streamlit
import json

ann_plain.save("churn_ann.keras")
json.dump({"columns": list(xtr.columns), "num": num, "mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist()},
          open("churn_preprocess.json", "w"))

## Conclusion
I trained an artificial neural network (two hidden layers, 32 and 16 neurons) to predict which telecom customers will leave, and compared it with Logistic Regression. The customers who leave are only 26.5% of the data, so we looked at recall and AUC and not only at accuracy.